# AL-02 동선 최적화 엔진 — 통합 파이프라인 (1차 버전)

| 구분 | 내용 |
|---|---|
| 가중치 정책 | `ahp_v1_20260907` |
| 산정 방법 | AHP 쌍대비교 (Saaty 척도) + 고유벡터법 |
| 판단 근거 | K-POP 팬 여행 타겟의 성향 기준 |
| 일관성 | CR = 0.0032 (< 0.10, 통과) |

$$relevance = 0.6483 \\times artist\\_match + 0.2297 \\times category\\_fitness + 0.1220 \\times place\\_quality$$

**제거된 항목:** sentiment (0.20), fan_interest (0.10), trust/evidence_tier, rating (0.05)

## ⚠ event_no vs matrix index
- `user_input`의 `concert.event_no`는 **DB event_no** (예: 1014)
- `matrix`, `names`, `stays`는 **events 리스트 인덱스** (0~N-1)
- `AL02Pipeline.run()`에서 자동 변환

## 1. 정책 상수

In [ ]:
import numpy as np
import itertools
import time
import timeit
import math
from datetime import datetime, timedelta

SCORING_POLICY = {
    "policy_version": "ahp_v1_20260907",
    "decision_method": "AHP pairwise comparison (Saaty scale) + eigenvalue method",
    "weights": {"artist_match": 0.6483, "category_fitness": 0.2297, "place_quality": 0.1220},
    "pairwise_judgments": {
        ("artist_match", "category_fitness"): 3.0,
        ("artist_match", "place_quality"): 5.0,
        ("category_fitness", "place_quality"): 2.0,
    },
    "consistency": {"lambda_max": 3.0037, "CI": 0.0018, "RI": 0.58, "CR": 0.0032},
    "rationale": "K-POP 팬 여행 타겟의 성향 기준",
    "revision_trigger": ["total_score 산식 변경", "장소-멤버 연결 테이블 추가", "실사용자 데이터 축적 시"],
}

ROUTE_POLICY = {
    "CONCERT_BUFFER_MIN": 150, "MAX_PER_DAY_NORMAL": 5,
    "MAX_PER_DAY_CONCERT": 4, "CONCERT_DURATION_MIN": 90,
    "DAY_START_MIN": 540, "DAY_END_MIN": 1260,
}

STAY_MAP = {
    "콘서트/팬미팅": 120, "팬사인회": 90, "공연/행사": 120,
    "팝업": 60, "기타": 40, "생일 카페": 60,
    "성지 디저트/카페": 60, "성지 음식점": 60, "팝업/굿즈": 45,
    "기타 성지": 40, "문화/유적지": 90, "여행지": 90, "쇼핑": 60,
}

CTG_TYPE = {1: "행사", 2: "성지", 3: "관광 명소"}
ALLOWED_PREFERENCE_CTG_TYPES = [2, 3]
ALLOWED_OP_STATUS = [1, 2]

print("정책 상수 설정 완료")
print(f"  가중치: {SCORING_POLICY['weights']}")
print(f"  CR: {SCORING_POLICY['consistency']['CR']} (통과)")

## 2. S2 — 추천 점수 (AHP 3-기준 가중합)

### ⚠ v4 정정 (2026-09-09) — v3에서 잘못 확장한 부분 되돌림
v3에서 "선택 안 한 같은 그룹 멤버의 개별 이벤트"까지 0.7로 끌어올렸었으나,
**이건 잘못된 판단이었다.** 실제 원칙은:

1. **선택 멤버 본인의 이벤트가 최우선(1.0)**
2. **정 없으면(부족하면) 그룹 전체 태그 콘텐츠로 폴백(0.7)** — S3가 relevance 내림차순으로
   채우기 때문에, 1.0이 항상 먼저 배정되고 0.7은 자리가 남을 때만 채워지는 구조 자체가 이미
   "우선순위 폴백"이다.
3. **선택 안 한 다른 멤버의 개별 이벤트는 무관(0.0)** — "그룹 전체" 공식 콘텐츠가 아니라
   "그룹 내 다른 개인" 콘텐츠이므로 폴백 대상이 아니다.

`build_selected_group_nos()`(선택 멤버의 소속 그룹을 미리 알아내는 부분)는 그대로 유지한다 —
멤버를 개별 선택한 경우에도 "그룹 전체" 태그 콘텐츠(2번 폴백)가 정상적으로 걸리게 하려면 여전히 필요하다.
달라진 건 **누구에게 0.7을 주느냐** — 그룹 전체 태그 이벤트에만 주고, 다른 개별 멤버 이벤트에는 주지 않는다.

In [ ]:
def _roc(rank, n):
    return sum(1.0 / k for k in range(rank, n + 1)) / n

def _to_min(t):
    h, m = map(int, str(t).split(":"))
    return h * 60 + m

def _to_str(m):
    return f"{int(m) // 60:02d}:{int(m) % 60:02d}"

def build_selected_group_nos(selected_group_nos_explicit, selected_artist_nos, artist_group_map):
    """선택 멤버의 소속 그룹을 알아내는 용도로만 사용한다(폴백 대상 판정용).
    명시적으로 고른 그룹(그룹 전체 선택 모드) + 개별 선택 멤버들의 소속 그룹을 합집합으로 만든다.
    이렇게 해야 "멤버 N명만 개별 선택"한 경우에도, 그 멤버가 속한 그룹의 "그룹 전체" 태그
    콘텐츠(예: 그룹 팬미팅)가 폴백 후보로 정상 매칭된다.

    artist_group_map: {artist_no: artist_group_no} — S0 단계에서 DB(artist 테이블)로
    선택 멤버들의 소속 그룹을 한 번에 조회해 만들어 전달한다.
    """
    groups = set(int(g) for g in (selected_group_nos_explicit or []))
    for a in (selected_artist_nos or []):
        g = (artist_group_map or {}).get(int(a))
        if g is not None:
            groups.add(int(g))
    return groups

def artist_match(event_artist_no, event_artist_group_no,
                selected_artist_nos, selected_group_nos):
    """1.0=선택 멤버 본인 이벤트(최우선). 0.7=그룹 전체 태그 이벤트(정 없으면 폴백).
    0.0=그 외 전부(선택 안 한 다른 멤버의 개별 이벤트 포함 — 그룹 전체 콘텐츠가 아니므로 폴백 대상 아님).
    다중 멤버 선택 시 max 사용.

    v4 정정: v3에서 "선택 안 한 같은 그룹 멤버 이벤트"에도 0.7을 주도록 확장했었으나 되돌림.
    "선택 멤버 우선, 부족하면 그룹 전체로 폴백"이지 "부족하면 같은 그룹 아무나로 폴백"이 아니기 때문.
    실 DB는 artist_no/artist_group_no 중 하나만 채워지므로(둘 다 채워지는 행 없음), 아래 두 조건은
    사실상 겹치지 않는다.
    """
    if selected_artist_nos and event_artist_no is not None:
        if int(event_artist_no) in [int(a) for a in selected_artist_nos]:
            return 1.0
        return 0.0  # 선택 안 한 다른 멤버의 개별 이벤트 — 그룹 전체 콘텐츠가 아니므로 폴백 대상 아님
    if selected_group_nos and event_artist_group_no is not None:
        if int(event_artist_group_no) in [int(g) for g in selected_group_nos]:
            return 0.7
    return 0.0

def category_fitness(event_ctg_no, selected_ctg_nos):
    # 0.3~1.0. ROC + 하한 보정.
    MISS = 0.3
    if not selected_ctg_nos or event_ctg_no not in selected_ctg_nos:
        return MISS
    n, rank = len(selected_ctg_nos), selected_ctg_nos.index(event_ctg_no) + 1
    return round(MISS + (1 - MISS) * (_roc(rank, n) / _roc(1, n)), 4)

def place_quality(total_score, score_min=0.0, score_max=100.0):
    # total_score 정규화. DA팀 확인 완료(2026-09-09): 0~100점 만점.
    if total_score is None: return 0.0
    ts = float(total_score)
    if score_max == score_min: return 0.0
    return round(max(0.0, min(1.0, (ts - score_min) / (score_max - score_min))), 4)

def hard_filter(event):
    op = event.get("op_status_no")
    if op is None: return False
    return int(op) in ALLOWED_OP_STATUS

def calc_relevance(event, user_input, score_range=(0.0, 100.0), artist_group_map=None):
    w = SCORING_POLICY["weights"]
    selected_artist_nos = user_input.get("artist_nos", [])
    selected_group_nos = build_selected_group_nos(
        user_input.get("group_nos", []), selected_artist_nos, artist_group_map)
    am = artist_match(event.get("artist_no"), event.get("artist_group_no"),
                      selected_artist_nos, selected_group_nos)
    cf = category_fitness(event.get("ctg_no"), user_input.get("ctg_nos", []))
    pq = place_quality(event.get("total_score"), score_range[0], score_range[1])
    score = w["artist_match"] * am + w["category_fitness"] * cf + w["place_quality"] * pq
    return {"relevance": round(score, 4),
            "artist_match": am, "category_fitness": cf, "place_quality": pq}

# 테스트 (v4: 선택 멤버 우선 / 그룹 전체 폴백 / 다른 개별멤버 무관 — 3단 검증)
artist_group_map_test = {101: 1, 102: 1, 999: 2}  # 101,102=그룹1 소속, 999=그룹2 소속
user = {"artist_nos": [101], "group_nos": [], "ctg_nos": [5, 8, 11]}  # 멤버 101만 개별 선택(그룹 미선택)

test_ev = {"event_no": 42, "artist_no": 101, "artist_group_no": None,
           "ctg_no": 5, "op_status_no": 2, "total_score": 95,
           "event_nm": "지민 생일카페(선택 멤버 본인)", "ctg_nm": "생일 카페"}
r = calc_relevance(test_ev, user, artist_group_map=artist_group_map_test)
print(f"[1순위] 지민(101, 선택멤버 본인) 생일카페: relevance={r['relevance']}, artist_match={r['artist_match']}  (기대: 1.0)")

test_ev_grp = {"event_no": 43, "artist_no": None, "artist_group_no": 1,
               "ctg_no": 5, "op_status_no": 2, "total_score": 95,
               "event_nm": "그룹1 팬미팅(그룹 전체 태그)", "ctg_nm": "생일 카페"}
r_grp = calc_relevance(test_ev_grp, user, artist_group_map=artist_group_map_test)
print(f"[2순위/폴백] 그룹1 팬미팅(그룹 전체 태그): relevance={r_grp['relevance']}, artist_match={r_grp['artist_match']}  (기대: 0.7)")

test_ev2 = {"event_no": 44, "artist_no": 102, "artist_group_no": None,
            "ctg_no": 5, "op_status_no": 2, "total_score": 95,
            "event_nm": "정국 생일카페(같은 그룹, 미선택 멤버)", "ctg_nm": "생일 카페"}
r2 = calc_relevance(test_ev2, user, artist_group_map=artist_group_map_test)
print(f"[무관] 정국(102, 같은 그룹·미선택) 생일카페: relevance={r2['relevance']}, artist_match={r2['artist_match']}  (기대: 0.0 -- v4에서 정정됨)")

test_ev3 = {"event_no": 45, "artist_no": 999, "artist_group_no": None,
            "ctg_no": 5, "op_status_no": 2, "total_score": 95,
            "event_nm": "다른 그룹 멤버 이벤트", "ctg_nm": "생일 카페"}
r3 = calc_relevance(test_ev3, user, artist_group_map=artist_group_map_test)
print(f"[무관] 다른 그룹(999) 이벤트: relevance={r3['relevance']}, artist_match={r3['artist_match']}  (기대: 0.0)")

print(f"\n카테고리: 1순위={category_fitness(5, [5,8,11])}, 미선택={category_fitness(2, [5,8,11])}")

## 3. S1 — 여행 프레임 설정

In [ ]:
def build_trip_frame(user_input):
    # S1: 여행 프레임. concert.event_no는 DB event_no.
    start = datetime.strptime(user_input["trip_start"], "%Y-%m-%d")
    end = datetime.strptime(user_input["trip_end"], "%Y-%m-%d")
    n_days = (end - start).days + 1
    date_list = [(start + timedelta(days=i)).strftime("%Y-%m-%d")
                 for i in range(n_days)]

    concert = user_input.get("concert")
    concert_day_idx = None
    concert_event_no = None
    concert_start_min = None
    concert_duration = ROUTE_POLICY["CONCERT_DURATION_MIN"]

    if concert:
        if concert.get("event_date") in date_list:
            concert_day_idx = date_list.index(concert["event_date"])
        concert_event_no = concert.get("event_no")
        concert_start_min = _to_min(concert.get("start_time", "19:00"))
        concert_duration = concert.get("duration_min", ROUTE_POLICY["CONCERT_DURATION_MIN"])

    lodging = user_input.get("lodging", {})

    # ── 출발핀/도착핀 (핀으로 찍는 지점) ────────────────────────────
    # start_pin: Day 1 출발 지점 (예: 공항). 없으면 숙소에서 출발.
    # end_pin:   마지막 날 완료 지점 (예: 공항). 없으면 숙소로 복귀.
    start_pin = user_input.get("start_pin")   # {"latitude":.., "longitude":..} 또는 None
    end_pin = user_input.get("end_pin")

    return {
        "n_days": n_days, "date_list": date_list,
        "concert_day_idx": concert_day_idx,
        "concert_event_no": concert_event_no,
        "concert_start_min": concert_start_min,
        "concert_duration": concert_duration,
        "day_start_min": _to_min(user_input.get("day_start_time", "09:00")),
        "day_end_min": _to_min(user_input.get("day_end_time", "21:00")),
        "depot_lat": lodging.get("latitude"), "depot_lon": lodging.get("longitude"),
        "start_pin": start_pin, "end_pin": end_pin,
    }

print("S1 함수 정의 완료 (출발핀/도착핀 반영)")

## 4. S4 — 하루 방문 순서 최적화 (완전탐색)

⚠ `concert_idx`는 **matrix 인덱스** (0~N-1). event_no가 아님!

In [ ]:
def build_cache(events):
    names = [e["event_nm"] for e in events]
    stays = np.array([STAY_MAP.get(e.get("ctg_nm", "기타"), 40)
                      for e in events], dtype=np.int32)
    return names, stays


def haversine_min(lat1, lon1, lat2, lon2, speed_kmh=25.0):
    """두 좌표 간 이동시간(분) 근사. 실제로는 카카오맵 API/캐시로 대체.
    speed_kmh: 도심 평균 이동속도 가정(차량 25km/h)."""
    import math as _m
    R = 6371.0
    p1, p2 = _m.radians(lat1), _m.radians(lat2)
    dphi = _m.radians(lat2 - lat1)
    dlmb = _m.radians(lon2 - lon1)
    a = _m.sin(dphi/2)**2 + _m.cos(p1)*_m.cos(p2)*_m.sin(dlmb/2)**2
    dist_km = 2 * R * _m.asin(_m.sqrt(a))
    return int(round(dist_km / speed_kmh * 60))


def augment_matrix(matrix, events, extra_points):
    """POI 이동시간 행렬 끝에 임시 노드(숙소/출발핀/도착핀)를 추가한다.
    extra_points: [{"lat":.., "lon":..}, ...] 순서대로 인덱스 N, N+1, ... 부여.
    반환: (확장행렬, [추가된 노드의 인덱스들])
    좌표가 없는(None) 지점은 이동시간 0(=제약 없음)으로 채운다.
    """
    N = matrix.shape[0]
    k = len(extra_points)
    if k == 0:
        return matrix, []
    new_size = N + k
    aug = np.zeros((new_size, new_size), dtype=matrix.dtype)
    aug[:N, :N] = matrix

    # 각 POI의 좌표
    poi_ll = [(e.get("event_lat"), e.get("event_lon")) for e in events]

    added_idx = list(range(N, N + k))
    for offset, pt in enumerate(extra_points):
        gi = N + offset
        plat, plon = pt.get("lat"), pt.get("lon")
        for j in range(N):
            jlat, jlon = poi_ll[j]
            if plat is None or plon is None or jlat is None or jlon is None:
                t = 0   # 좌표 없으면 이동시간 0 (depot 미지정과 동일 취급)
            else:
                t = haversine_min(plat, plon, jlat, jlon)
            aug[gi, j] = t
            aug[j, gi] = t
    # 추가 노드끼리의 거리도 채움(같은 날 start=end일 때 등)
    for a in range(k):
        for b in range(k):
            if a == b: continue
            ga, gb = N + a, N + b
            la, lo = extra_points[a].get("lat"), extra_points[a].get("lon")
            lb, ob = extra_points[b].get("lat"), extra_points[b].get("lon")
            if None in (la, lo, lb, ob):
                aug[ga, gb] = 0
            else:
                aug[ga, gb] = haversine_min(la, lo, lb, ob)
    return aug, added_idx


def s4_solve(day_indices, start_idx, end_idx, matrix, names, stays,
             concert_idx=None, concert_start=None,
             concert_duration=None, concert_buffer=None,
             day_start=None, day_end=None, max_places=None):
    """S4: 완전탐색. 공연은 마지막 고정.
    start_idx: 그날의 출발 depot(matrix 인덱스). None이면 출발 이동 0.
    end_idx:   그날의 도착 depot(matrix 인덱스). None이면 복귀 이동 0.
    concert_idx는 matrix 인덱스 (0~N-1)."""
    buf = concert_buffer if concert_buffer is not None else ROUTE_POLICY["CONCERT_BUFFER_MIN"]
    ds = day_start if day_start is not None else ROUTE_POLICY["DAY_START_MIN"]
    de = day_end if day_end is not None else ROUTE_POLICY["DAY_END_MIN"]
    mp = max_places if max_places is not None else ROUTE_POLICY["MAX_PER_DAY_NORMAL"]
    cd = concert_duration if concert_duration is not None else ROUTE_POLICY["CONCERT_DURATION_MIN"]

    non_concert = [int(i) for i in day_indices if int(i) != concert_idx]
    if len(non_concert) > mp:
        non_concert = non_concert[:mp]

    deadline = None
    if concert_idx is not None:
        deadline = (concert_start - buf) if concert_start is not None else (de - buf)

    stay_arr = stays.copy()
    if concert_idx is not None:
        stay_arr[concert_idx] = cd

    best_order, best_cost, best_sched = None, float("inf"), []
    for perm in itertools.permutations(non_concert):
        order = list(perm)
        if concert_idx is not None:
            order.append(concert_idx)

        # 출발/도착 depot을 앞뒤에 붙여 이동시간 계산
        # start_idx/end_idx가 None이면 해당 구간 이동 0
        seq = order[:]
        head = [start_idx] if start_idx is not None else []
        tail = [end_idx] if end_idx is not None else []
        route = head + seq + tail
        if len(route) >= 2:
            route_arr = np.array(route, dtype=np.int32)
            leg_times = matrix[route_arr[:-1], route_arr[1:]]
        else:
            leg_times = np.zeros(0, dtype=np.int64)

        # 방문지별 도착 이동시간: head가 있으면 leg_times[0]이 start→첫방문
        # 없으면 첫 방문의 이동시간 0
        ct = ds; tt = 0; sched = []; ok = True
        # 각 방문지 k에 대한 진입 이동시간 인덱스 계산
        for k, idx in enumerate(order):
            if head:
                tr = int(leg_times[k])          # start→order[0], order[0]→order[1] ...
            else:
                tr = int(leg_times[k-1]) if k > 0 else 0
            ar = ct + tr
            st = int(stay_arr[idx]); dp = ar + st
            if concert_idx is not None and idx == concert_idx:
                if ar > deadline: ok = False; break
            if dp > de: ok = False; break
            sched.append({"idx": idx, "name": names[idx], "travel": tr,
                          "arrive": _to_str(ar), "depart": _to_str(dp), "stay": st})
            tt += tr; ct = dp
        if not ok: continue
        # 마지막 방문 → 도착 depot 이동시간 추가
        if tail and len(order) > 0:
            tt += int(leg_times[-1])
        if tt < best_cost:
            best_cost = tt; best_order = order; best_sched = sched

    return {"order": best_order, "cost": int(best_cost) if best_order else 99999,
            "schedule": best_sched}


# 성능 테스트 (출발/도착 depot 지정)
np.random.seed(42)
tm = np.random.randint(5, 60, size=(10, 10))
tm = (tm + tm.T) // 2
np.fill_diagonal(tm, 0)
tn = [f"장소{i}" for i in range(10)]
ts = np.full(10, 60, dtype=np.int32)

print("S4 성능 (start/end depot 분리):")
print(f"{'n':>4} {'순열':>8} {'시간':>12}")
print("-" * 28)
for n in range(3, 8):
    ti = list(range(n))
    times = []
    for _ in range(7):
        t0 = time.perf_counter()
        s4_solve(ti, 0, 0, tm, tn, ts)   # start=end=0 (숙소 왕복)
        times.append((time.perf_counter() - t0) * 1000)
    print(f"{n:>4} {math.factorial(n):>8,} {min(times):>10.2f}ms")
print("\nS4 함수 정의 완료 (출발/도착 depot 분리)")


def s4_solve_fallback(day_indices, start_idx, end_idx, matrix, names, stays,
                      score_map, concert_idx=None, concert_start=None,
                      concert_duration=None, day_start=None, day_end=None):
    """S4 완전탐색이 시간 초과(99999)로 실패하면,
    relevance 최저 장소부터 1개씩 제거하며 유효한 동선이 나올 때까지 재시도.
    공연(concert_idx)은 절대 제거하지 않는다(하드 제약).
    반환: (result, dropped_list) — dropped_list는 제외된 장소 인덱스."""
    places = [int(p) for p in day_indices]
    dropped = []
    while places:
        result = s4_solve(places, start_idx, end_idx, matrix, names, stays,
                          concert_idx=concert_idx, concert_start=concert_start,
                          concert_duration=concert_duration,
                          day_start=day_start, day_end=day_end,
                          max_places=len(places))
        if result["order"] is not None and result["cost"] < 99999:
            return result, dropped
        # 실패 → 공연 제외한 것 중 relevance 최저 제거
        removable = [p for p in places if p != concert_idx]
        if not removable:
            break  # 공연만 남았는데도 실패 → 포기
        worst = min(removable, key=lambda p: score_map.get(p, 0.0))
        places.remove(worst)
        dropped.append(worst)
    return {"order": None, "cost": 99999, "schedule": []}, dropped


print("s4_solve_fallback 정의 완료 (시간 초과 시 자동 축소)")

## 5. S3 — 날짜별 장소 배정 (그리디 + 로컬서치)

⚠ 안전 체크는 반드시 **내부 루프 안**에!

In [ ]:
def day_depots(d, n_days, frame):
    """그날의 (start_idx, end_idx)를 반환.
    depot 인덱스는 augment_matrix로 matrix 끝에 추가된 노드 인덱스를 사용:
      frame["depot_idx"]      = 숙소
      frame["start_pin_idx"]  = Day1 출발핀 (없으면 None)
      frame["end_pin_idx"]    = 마지막날 도착핀 (없으면 None)
    규칙:
      Day 0(첫날): start = 출발핀(있으면) else 숙소 / end = 숙소
      마지막 날:   start = 숙소 / end = 도착핀(있으면) else 숙소
      중간 날:     start = 숙소 / end = 숙소
    """
    depot = frame.get("depot_idx")
    spin = frame.get("start_pin_idx")
    epin = frame.get("end_pin_idx")

    if d == 0:
        start = spin if spin is not None else depot
    else:
        start = depot
    if d == n_days - 1:
        end = epin if epin is not None else depot
    else:
        end = depot

    # ── 날짜별 출발지 override (기본 숙소, 예외적으로 유저가 변경) ──────
    # frame["day_start_override"] = {날짜인덱스: matrix_인덱스}
    # UI: SC-02c 동선 수정 화면에서 "이 날 출발지 변경" 시 설정됨.
    # 첫날 출발핀/마지막날 도착핀과 별개로, 중간날 출발지를 바꾸는 용도.
    overrides = frame.get("day_start_override", {})
    if d in overrides and overrides[d] is not None:
        start = overrides[d]
    return start, end


def _day_cost(day_places, start_idx, end_idx, concert_idx, matrix, names, stays,
              concert_start, concert_duration):
    if not day_places: return 0
    r = s4_solve(day_places, start_idx, end_idx, matrix, names, stays,
                 concert_idx=concert_idx, concert_start=concert_start,
                 concert_duration=concert_duration)
    return r["cost"]


def s3_greedy(candidates, frame, matrix, names, stays, concert_matrix_idx, W_rel=30.0):
    score_map = {int(i): float(s) for i, s in candidates}
    n_days = frame["n_days"]
    concert_day = frame["concert_day_idx"]

    day_plans = {d: [] for d in range(n_days)}
    if concert_day is not None and concert_matrix_idx is not None:
        day_plans[concert_day].append(int(concert_matrix_idx))
    assigned = set()
    if concert_matrix_idx is not None:
        assigned.add(int(concert_matrix_idx))

    for idx, score in sorted(candidates, key=lambda x: x[1], reverse=True):
        idx = int(idx)
        if idx in assigned: continue
        best_day, best_gain = None, float("inf")
        for d in range(n_days):
            is_c = (d == concert_day)
            limit = ROUTE_POLICY["MAX_PER_DAY_CONCERT"] if is_c else ROUTE_POLICY["MAX_PER_DAY_NORMAL"]
            if len(day_plans[d]) >= limit: continue
            c_idx = int(concert_matrix_idx) if is_c else None
            c_start = frame["concert_start_min"] if is_c else None
            c_dur = frame["concert_duration"] if is_c else None
            s_idx, e_idx = day_depots(d, n_days, frame)
            before = _day_cost(day_plans[d], s_idx, e_idx, c_idx, matrix, names, stays, c_start, c_dur)
            after = _day_cost(day_plans[d]+[idx], s_idx, e_idx, c_idx, matrix, names, stays, c_start, c_dur)
            if after >= 99999: continue
            # W_rel: 이동시간 증가 - relevance 보너스 (클수록 취향 우선)
            gain = (after - before) - W_rel * score_map.get(idx, 0.0)
            if gain < best_gain: best_gain = gain; best_day = d
        if best_day is not None:
            day_plans[best_day].append(idx); assigned.add(idx)
    return day_plans


def s3_local_search(day_plans, frame, matrix, names, stays, concert_matrix_idx, candidates=None, W_rel=30.0, max_iter=10):
    score_map = {int(i): float(s) for i, s in (candidates or [])}
    n_days = frame["n_days"]
    concert_day = frame["concert_day_idx"]
    plans = {d: [int(p) for p in v] for d, v in day_plans.items()}

    def day_cost(d):
        c_idx = int(concert_matrix_idx) if d == concert_day else None
        c_start = frame["concert_start_min"] if d == concert_day else None
        c_dur = frame["concert_duration"] if d == concert_day else None
        s_idx, e_idx = day_depots(d, n_days, frame)
        return _day_cost(plans[d], s_idx, e_idx, c_idx, matrix, names, stays, c_start, c_dur)

    cost_cache = {d: day_cost(d) for d in range(n_days)}
    rel_cache = {d: sum(score_map.get(p, 0.0) for p in plans[d]) for d in range(n_days)}
    def comp(cost, rel): return cost - W_rel * rel
    cur = sum(comp(cost_cache[d], rel_cache[d]) for d in range(n_days))
    improved, it = True, 0
    while improved and it < max_iter:
        improved = False; it += 1

        # Shift
        for da in range(n_days):
            for poi in list(plans[da]):
                if poi == concert_matrix_idx: continue
                for db in range(n_days):
                    if da == db: continue
                    if poi not in plans[da]: continue
                    is_c = (db == concert_day)
                    lim = ROUTE_POLICY["MAX_PER_DAY_CONCERT"] if is_c else ROUTE_POLICY["MAX_PER_DAY_NORMAL"]
                    if len(plans[db]) >= lim: continue
                    plans[da].remove(poi); plans[db].append(poi)
                    nra = rel_cache[da] - score_map.get(poi, 0.0)
                    nrb = rel_cache[db] + score_map.get(poi, 0.0)
                    na, nb = day_cost(da), day_cost(db)
                    nc = (cur - comp(cost_cache[da], rel_cache[da]) - comp(cost_cache[db], rel_cache[db])
                          + comp(na, nra) + comp(nb, nrb))
                    if nc < cur:
                        cur = nc; cost_cache[da], cost_cache[db] = na, nb
                        rel_cache[da], rel_cache[db] = nra, nrb; improved = True
                    else:
                        plans[db].remove(poi); plans[da].append(poi)

        # Swap
        for da in range(n_days):
            for db in range(da+1, n_days):
                for pa in list(plans[da]):
                    if pa == concert_matrix_idx: continue
                    for pb in list(plans[db]):
                        if pb == concert_matrix_idx: continue
                        if pa not in plans[da]: continue
                        if pb not in plans[db]: continue
                        plans[da].remove(pa); plans[db].remove(pb)
                        plans[da].append(pb); plans[db].append(pa)
                        nra = rel_cache[da] - score_map.get(pa,0.0) + score_map.get(pb,0.0)
                        nrb = rel_cache[db] - score_map.get(pb,0.0) + score_map.get(pa,0.0)
                        na, nb = day_cost(da), day_cost(db)
                        nc = (cur - comp(cost_cache[da], rel_cache[da]) - comp(cost_cache[db], rel_cache[db])
                              + comp(na, nra) + comp(nb, nrb))
                        if nc < cur:
                            cur = nc; cost_cache[da], cost_cache[db] = na, nb
                            rel_cache[da], rel_cache[db] = nra, nrb; improved = True
                        else:
                            plans[da].remove(pb); plans[db].remove(pa)
                            plans[da].append(pa); plans[db].append(pb)
    total_travel = sum(cost_cache.values())
    return plans, total_travel

print("S3 함수 정의 완료 (날짜별 start/end depot + override)")

## 6. S5 — 검증

In [ ]:
def verify(day_plans, frame, concert_matrix_idx, s4_results=None):
    checks = {}
    all_pois = [p for v in day_plans.values() for p in v]
    checks["중복 없음"] = len(all_pois) == len(set(all_pois))
    for d, v in day_plans.items():
        is_c = (d == frame["concert_day_idx"])
        lim = ROUTE_POLICY["MAX_PER_DAY_CONCERT"] if is_c else ROUTE_POLICY["MAX_PER_DAY_NORMAL"]
        checks[f"Day {d+1} 상한 {lim}"] = len(v) <= lim
    if concert_matrix_idx is not None:
        cd = frame["concert_day_idx"]
        checks["공연 배정"] = int(concert_matrix_idx) in day_plans.get(cd, [])
        if s4_results and cd is not None and s4_results.get(cd):
            sched = s4_results[cd]
            if sched and sched["schedule"]:
                checks["공연 마지막"] = (sched["schedule"][-1]["idx"] == int(concert_matrix_idx))
    return {"passed": all(checks.values()), "checks": checks}

print("S5 함수 정의 완료")

## 7. 통합 파이프라인

⚠ `concert.event_no` (DB ID) → `concert_matrix_idx` (배열 인덱스) 변환이
`run()` 내부에서 자동 수행됩니다.

### v3 변경 사항 (2026-09-09)
- `AL02Pipeline(artist_group_map=...)` — S0 단계에서 조회한 `{artist_no: artist_group_no}`를 받아
  candidates 점수 계산(`calc_relevance`)에 그대로 전달할 수 있게 함(팬덤 그룹 매칭 보정, 2절 참고)
- `run()`의 `schedule` 각 항목에 `relevance` 필드 추가 — 기존엔 누락되어 있었음
- `run_abc()`의 `rel_sum` 죽은 코드(`for s in []`) 제거, `total_relevance`/`avg_relevance`
  실제 집계로 교체 — A/B/C 동선 비교 화면(SC-03)에서 취향 반영도 지표로 사용 가능

In [ ]:
# W_rel 프로파일 (A/B/C 동선)
# ⚠ 주의: 15/30/60 값은 실측 전 임시값이다. 실제 relevance 분포와
#   이동시간 분포로 재보정해야 함. A/B/C 선택 UX는 확정, 수치는 미확정.
WREL_PROFILES = {
    "A": {"W_rel": 15, "label": "여유 우선 (이동 최소)"},
    "B": {"W_rel": 30, "label": "균형 (기본)"},
    "C": {"W_rel": 60, "label": "많이 보기 (취향 우선)"},
}


class AL02Pipeline:
    # AL-02 전체 파이프라인.
    # user_input의 concert.event_no는 DB의 event_no.
    # depot(숙소)/출발핀/도착핀/날짜별 출발지 override는 augment_matrix로
    #   matrix 끝에 임시 노드로 추가된다.

    def __init__(self, score_range=(0.0, 100.0)):
        self.score_range = score_range
        self.policy = SCORING_POLICY
        # artist_group_map은 여기서 쓰지 않는다 — candidates 점수(relevance)는 run() 호출 전에
        # calc_relevance()로 이미 계산되어 들어오므로, 그룹 매핑은 S0/candidate 준비 단계의 책임이다.

    def run(self, candidates, matrix, events, user_input, W_rel=30.0):
        frame = build_trip_frame(user_input)
        names, stays = build_cache(events)
        N = matrix.shape[0]
        score_map = {int(idx): float(s) for idx, s in candidates}

        # ── depot/핀/override를 matrix에 임시 노드로 추가 ────────────
        extra = [{"lat": frame["depot_lat"], "lon": frame["depot_lon"]}]  # 숙소 = index N
        depot_idx = N
        start_pin_idx = None
        end_pin_idx = None
        sp = frame.get("start_pin")
        ep = frame.get("end_pin")
        if sp:
            extra.append({"lat": sp.get("latitude"), "lon": sp.get("longitude")})
            start_pin_idx = N + len(extra) - 1
        if ep:
            extra.append({"lat": ep.get("latitude"), "lon": ep.get("longitude")})
            end_pin_idx = N + len(extra) - 1

        # 날짜별 출발지 override: user_input["day_start_override"] = {날짜idx: {lat,lon}}
        # SC-02c 동선 수정 화면에서 중간날 출발지를 바꿀 때 사용.
        override_map = {}
        raw_override = user_input.get("day_start_override", {})
        for d, pt in raw_override.items():
            if pt and pt.get("latitude") is not None:
                extra.append({"lat": pt.get("latitude"), "lon": pt.get("longitude")})
                override_map[int(d)] = N + len(extra) - 1

        aug_matrix, _ = augment_matrix(matrix, events, extra)
        frame["depot_idx"] = depot_idx
        frame["start_pin_idx"] = start_pin_idx
        frame["end_pin_idx"] = end_pin_idx
        frame["day_start_override"] = override_map

        pad = len(extra)
        pad_names = ["숙소"]
        if sp: pad_names.append("출발핀")
        if ep: pad_names.append("도착핀")
        pad_names += [f"출발지(day{d})" for d in override_map]
        names = names + pad_names
        stays = np.concatenate([stays, np.zeros(pad, dtype=stays.dtype)])
        matrix = aug_matrix

        # ── event_no → matrix 인덱스 변환 ──────────────────
        concert_matrix_idx = None
        concert_event_no = frame.get("concert_event_no")
        if concert_event_no is not None:
            for ii, ev in enumerate(events):
                if ev.get("event_no") == concert_event_no:
                    concert_matrix_idx = ii
                    break
            if concert_matrix_idx is None:
                raise ValueError(
                    f"공연 event_no={concert_event_no}가 events에서 발견되지 않습니다."
                )

        # S3: 날짜 배정 (W_rel 반영)
        day_plans = s3_greedy(candidates, frame, matrix, names, stays,
                              concert_matrix_idx, W_rel=W_rel)
        day_plans, total_cost = s3_local_search(
            day_plans, frame, matrix, names, stays, concert_matrix_idx,
            candidates=candidates, W_rel=W_rel)

        # S4: 각 날짜 순서 확정 (시간 초과 시 자동 축소)
        days_output = []
        s4_results = {}
        all_dropped = []
        n_days = frame["n_days"]
        for d in range(n_days):
            is_c = (d == frame["concert_day_idx"])
            c_idx = int(concert_matrix_idx) if is_c else None
            c_start = frame["concert_start_min"] if is_c else None
            c_dur = frame["concert_duration"] if is_c else None
            s_idx, e_idx = day_depots(d, n_days, frame)

            result, dropped = s4_solve_fallback(
                day_plans[d], s_idx, e_idx, matrix, names, stays, score_map,
                concert_idx=c_idx, concert_start=c_start,
                concert_duration=c_dur,
                day_start=frame["day_start_min"],
                day_end=frame["day_end_min"],
            )
            s4_results[d] = result
            # 제외된 장소는 day_plans에서도 빼고 기록
            for p in dropped:
                if p in day_plans[d]:
                    day_plans[d].remove(p)
                all_dropped.append({"day_index": d, "event_no": events[p].get("event_no"),
                                    "event_nm": events[p].get("event_nm")})

            schedule = []
            for s in result["schedule"]:
                ev = events[s["idx"]]
                schedule.append({
                    "event_no": ev.get("event_no"),
                    "event_nm": s["name"],
                    "ctg_nm": ev.get("ctg_nm"),
                    "travel_from_prev_min": s["travel"],
                    "arrive": s["arrive"],
                    "depart": s["depart"],
                    "stay_min": s["stay"],
                    "is_concert": (s["idx"] == c_idx),
                    # v3 추가 — score_map(candidates 점수)에서 그대로 꺼내 응답에 노출.
                    # 프론트가 "왜 이 장소가 추천됐는지"(취향 반영도)를 보여줄 수 있게 함.
                    "relevance": round(score_map.get(s["idx"], 0.0), 4),
                })

            # start_point 표시: override > 출발핀 > 숙소
            if d in override_map and s_idx == override_map[d]:
                sp_label = "사용자 지정 출발지"
            elif d == 0 and s_idx == start_pin_idx:
                sp_label = "출발핀"
            else:
                sp_label = "숙소"

            days_output.append({
                "day_index": d,
                "date": frame["date_list"][d],
                "is_concert_day": is_c,
                "start_point": sp_label,
                "end_point": ("도착핀" if (d == n_days - 1 and e_idx == end_pin_idx) else "숙소"),
                "n_places": len(day_plans[d]),
                "travel_minutes": result["cost"] if result["order"] else 0,
                "dropped_for_time": [p for p in dropped],  # 시간초과로 제외된 장소
                "schedule": schedule,
            })

        verification = verify(day_plans, frame, concert_matrix_idx, s4_results)
        total_places = len([p for v in day_plans.values() for p in v])

        return {
            "scoring_policy": SCORING_POLICY["policy_version"],
            "wrel_used": W_rel,
            "summary": {
                "total_places": total_places,
                "total_travel_minutes": sum(
                    d["travel_minutes"] for d in days_output),
                "validation_passed": verification["passed"],
                "validation_details": verification["checks"],
                "dropped_for_time": all_dropped,  # 시간 초과로 빠진 장소 전체
            },
            "days": days_output,
        }


def run_abc(pipeline, candidates, matrix, events, user_input):
    """A/B/C 세 동선을 한 번에 생성. 유저가 SC-03에서 비교/선택.

    v3 수정: 기존 코드는 `for s in []`로 인해 rel_sum이 항상 0으로 계산되고
    결과에 쓰이지도 않는 죽은 코드였음(relevance가 schedule에 없었기 때문).
    이제 schedule에 relevance가 실리므로(위 run() 패치) 실제로 집계해서
    total_relevance / avg_relevance를 결과에 포함한다.
    W_rel 자체는 "배치"에만 영향을 주고 "선택"엔 영향이 제한적이라(5.5절 알려진 한계),
    A/B/C 간 이 값이 크게 다르지 않을 수 있음 — 그 자체가 유의미한 관찰이 된다."""
    results = {}
    for key, prof in WREL_PROFILES.items():
        r = pipeline.run(candidates, matrix, events, user_input, W_rel=prof["W_rel"])
        all_sched = [s for d in r["days"] for s in d["schedule"]]
        total_rel = round(sum(s.get("relevance", 0.0) for s in all_sched), 4)
        avg_rel = round(total_rel / len(all_sched), 4) if all_sched else 0.0
        results[key] = {
            "label": prof["label"], "W_rel": prof["W_rel"],
            "total_places": r["summary"]["total_places"],
            "total_travel_minutes": r["summary"]["total_travel_minutes"],
            "total_relevance": total_rel,
            "avg_relevance": avg_rel,
            "result": r,
        }
    return results


print("통합 파이프라인 정의 완료 (W_rel A/B/C + 시간초과 자동축소 + 날짜별 출발지)")

## 8. AHP 가중치 재계산

In [ ]:
def recalculate_ahp_weights(judgments):
    criteria = sorted(set([k[0] for k in judgments] + [k[1] for k in judgments]))
    n = len(criteria)
    idx = {c: i for i, c in enumerate(criteria)}
    M = np.ones((n, n))
    for (r, c), v in judgments.items():
        M[idx[r], idx[c]] = v
        M[idx[c], idx[r]] = 1.0 / v
    eigvals, eigvecs = np.linalg.eig(M)
    max_i = np.argmax(eigvals.real)
    lam = eigvals.real[max_i]
    w = np.abs(eigvecs[:, max_i].real)
    w = w / w.sum()
    CI = (lam - n) / (n - 1)
    RI = {1: 0, 2: 0, 3: 0.58, 4: 0.90, 5: 1.12}.get(n, 1.12)
    CR = CI / RI if RI > 0 else 0.0
    return {"weights": {c: round(float(w[idx[c]]), 4) for c in criteria},
            "lambda_max": round(float(lam), 4),
            "CI": round(CI, 4), "CR": round(CR, 4), "consistent": CR < 0.10}

r = recalculate_ahp_weights(SCORING_POLICY["pairwise_judgments"])
print(f"가중치: {r['weights']}")
print(f"CR = {r['CR']} ({'통과' if r['consistent'] else '실패'})")

## 9. 전체 테스트 (15 POI, 3일, 공연 포함)

In [ ]:
np.random.seed(42)
N = 15
ctg_list = list(STAY_MAP.keys())
events = []
for i in range(N):
    events.append({
        "event_no": 1000+i, "event_nm": f"테스트 장소 {i+1}",
        "ctg_nm": ctg_list[i % len(ctg_list)],
        # v4 테스트 데이터: 3단 검증용으로 재구성
        #   i=0~4  (5개): 선택 멤버(101) 본인 개별 이벤트           -> 기대 1.0
        #   i=5~7  (3개): 그룹1 "전체" 태그 이벤트(artist_no 없음)  -> 기대 0.7 (폴백)
        #   i=8~10 (3개): 같은 그룹 다른 멤버(102) 개별 이벤트      -> 기대 0.0 (v4에서 정정된 부분)
        #   i=11~14(4개): 다른 그룹(999) 이벤트                     -> 기대 0.0
        "artist_no": (101 if i < 5 else (None if i < 8 else (102 if i < 11 else 999))),
        "artist_group_no": (1 if 5 <= i < 8 else None),
        "ctg_no": (i % 13) + 1, "op_status_no": 2,
        "total_score": 70 + (i*2) % 30,
    })

matrix = np.random.randint(5, 60, size=(N, N))
matrix = (matrix + matrix.T) // 2
np.fill_diagonal(matrix, 0)

# 사용자는 "그룹 전체"가 아니라 "멤버 101만" 개별 선택(그룹 미선택, 화면 UX와 동일 패턴)
user_input = {
    "trip_start": "2026-09-10", "trip_end": "2026-09-12",
    "artist_nos": [101], "group_nos": [], "ctg_nos": [5, 8, 11],
    "concert": {"event_no": 1014, "event_date": "2026-09-11", "start_time": "19:00"},
}

# S0 단계에서 DB(artist 테이블)로 미리 조회해뒀다고 가정하는 매핑.
# 101,102는 그룹1(선택 멤버 101과 같은 그룹) 소속, 999는 그룹2(무관) 소속.
artist_group_map = {101: 1, 102: 1, 999: 2}

candidates = []
for i, ev in enumerate(events):
    if not hard_filter(ev): continue
    score = calc_relevance(ev, user_input, artist_group_map=artist_group_map)
    candidates.append((i, score["relevance"]))

print(f"후보: {len(candidates)}개")
print("\n[v4 검증] 선택멤버 우선(1.0) / 그룹전체 폴백(0.7) / 다른개별멤버 무관(0.0) 3단 확인:")
selected_group_nos = build_selected_group_nos([], [101], artist_group_map)
for i, ev in enumerate(events[:11]):
    am = artist_match(ev["artist_no"], ev["artist_group_no"], [101], selected_group_nos)
    print(f"  event_no={ev['event_no']} artist_no={ev['artist_no']} artist_group_no={ev['artist_group_no']} -> artist_match={am}")

engine = AL02Pipeline()
t0 = time.perf_counter()
result = engine.run(candidates, matrix, events, user_input)
elapsed = (time.perf_counter() - t0) * 1000

print(f"실행 시간: {elapsed:.0f}ms (목표 3초 이하)")
print(f"검증: {'✅ 통과' if result['summary']['validation_passed'] else '❌ 실패'}")
print(f"총 이동: {result['summary']['total_travel_minutes']}분")
print(f"총 방문: {result['summary']['total_places']}곳")

for day in result["days"]:
    is_c = " 🎤공연일" if day["is_concert_day"] else ""
    print(f"\n  ── Day {day['day_index']+1} ({day['date']}){is_c} ──")
    print(f"     {day['n_places']}곳 | 이동 {day['travel_minutes']}분")
    for s in day["schedule"]:
        mark = " 🎤" if s["is_concert"] else "  "
        print(f"     {mark} {s['arrive']}~{s['depart']} ({s['stay_min']:3d}분) "
              f"relevance={s.get('relevance', 0.0):.2f}  {s['event_nm']}")

print("\n검증 상세:")
for check, passed in result["summary"]["validation_details"].items():
    icon = "✅" if passed else "❌"
    print(f"  {icon} {check}")

# v3 신규 — run_abc() 정상 동작 확인 (total_relevance/avg_relevance 죽은 코드였던 부분)
print("\n\n=== A/B/C 동선 비교 (run_abc, v3에서 relevance 집계 복구) ===")
abc = run_abc(engine, candidates, matrix, events, user_input)
for key, v in abc.items():
    print(f"  [{key}] {v['label']:16s} W_rel={v['W_rel']:3d}  "
          f"장소 {v['total_places']}곳  이동 {v['total_travel_minutes']}분  "
          f"total_relevance={v['total_relevance']}  avg_relevance={v['avg_relevance']}")